## ⚠️  DEPRECATED — Swift container creation
The Swift container below (`photoprism-proj03`) has been superseded by Chameleon native S3.
**Skip cells 1 and 2. Use the S3 setup cells below instead.**

---

# Provision Object Storage — proj03

Creates the `photoprism-proj03` Swift container on **CHI@TACC** for large data: raw images, DVC artifacts, MLflow model checkpoints.

**Run this ONCE.** The container survives VM teardowns.

**Resources created:**
- Swift container: `photoprism-proj03` on CHI@TACC

**Note:** Safe to re-run — `put_container` is idempotent.

In [ ]:
from chi import context
import chi, swiftclient

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@TACC")

print("Site: CHI@TACC")

In [ ]:
raise RuntimeError("Deprecated — the Swift container is no longer used. Skip to the S3 setup cells below.")
os_conn = chi.clients.connection()
token = os_conn.authorize()
storage_url = os_conn.object_store.get_endpoint()

swift_conn = swiftclient.Connection(
    preauthurl=storage_url,
    preauthtoken=token,
    retries=5,
)

# Idempotent — put_container returns 202 if container already exists
container_name = "photoprism-proj03"
swift_conn.put_container(container_name)
print(f"Container '{container_name}' is ready.")
print("\nBrowse at: https://chi.tacc.chameleoncloud.org/project/containers")

# --- Object Storage: Chameleon Native S3 (CHI@TACC) ---

MinIO has been removed. Chameleon's native S3 (via the Ceph RGW gateway at CHI@TACC) is used instead.

**Architecture:**
- Compute: KVM@TACC (your VM)
- Object storage: CHI@TACC (different logical site, accessed over network via S3 API)
- No extra service to run — Chameleon provides S3-compatible storage natively

**Endpoint:** `https://chi.tacc.chameleoncloud.org:7480`

**Bucket:** `training-module-proj03` — shared with your team. **Do NOT delete or recreate it.**

**Credentials:** Per-user EC2 keypairs (not shared). Each team member generates their own once.

## ONE-TIME: Generate S3 Credentials

Run this cell **once per user** to generate your personal EC2 access key + secret key.

> **Connect to CHI@TACC** (not KVM@TACC) — object storage lives there.
>
> ⚠️  **Save both values immediately. Treat the secret like a password.**
> **Clear this cell's output before committing.**

In [ ]:
# ONE-TIME: Generate EC2 credentials for Chameleon S3 (CHI@TACC)
# Replaced by Chameleon native S3 (CHI@TACC)
# Run once per user. Save the output — you cannot retrieve the secret again.

from chi import context
import chi

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@TACC")  # Must be CHI@TACC — object storage lives here

os_conn = chi.clients.connection()

# Direct REST call — per Chameleon documentation
# https://teaching-on-testbeds.github.io/data-persist-chi/
project_id = os_conn.current_project_id
identity_ep = os_conn.session.get_endpoint(service_type="identity", interface="public")
url = f"{identity_ep}/v3/users/{os_conn.current_user_id}/credentials/OS-EC2"

resp = os_conn.session.post(url, json={"tenant_id": project_id})
resp.raise_for_status()
ec2 = resp.json()["credential"]

print("=" * 60)
print("EC2 Credentials generated. Save these NOW.")
print("=" * 60)
print(f"AWS_ACCESS_KEY_ID     = {ec2['access']}")
print(f"AWS_SECRET_ACCESS_KEY = {ec2['secret']}")
print("=" * 60)
print("⚠️  Treat the secret like a password.")
print("⚠️  Clear this cell output before committing.")

## Configure rclone on the VM

Run this cell to SSH into your VM and configure rclone for Chameleon S3 access.

**Prerequisites:** VM must be running and SSH accessible. Fill in your floating IP and EC2 credentials below.

**Shared bucket — do not delete or recreate it.**

In [ ]:
# Task 2: Configure rclone on the VM for Chameleon S3 access
# Replaced by Chameleon native S3 (CHI@TACC)

import subprocess

# Fill these in before running
FLOATING_IP = "FILL-IN-YOUR-VM-FLOATING-IP"
EC2_ACCESS  = "FILL-IN-YOUR-EC2-ACCESS-KEY"   # from the credentials cell above
EC2_SECRET  = "FILL-IN-YOUR-EC2-SECRET-KEY"   # from the credentials cell above

if "FILL-IN" in FLOATING_IP or "FILL-IN" in EC2_ACCESS or "FILL-IN" in EC2_SECRET:
    raise ValueError("Fill in FLOATING_IP, EC2_ACCESS, and EC2_SECRET before running this cell.")

rclone_conf = f"""[chi_s3]
type = s3
provider = Ceph
access_key_id = {EC2_ACCESS}
secret_access_key = {EC2_SECRET}
endpoint = https://chi.tacc.chameleoncloud.org:7480
"""

# Write rclone config to VM
setup_cmd = f"""
mkdir -p ~/.config/rclone
cat > ~/.config/rclone/rclone.conf << 'RCLONE_EOF'
{rclone_conf}
RCLONE_EOF
echo "rclone config written."
"""

result = subprocess.run(
    ["ssh", "-o", "StrictHostKeyChecking=no", f"cc@{FLOATING_IP}", setup_cmd],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

# Verify connection — list buckets accessible to this user
# Shared bucket — do not delete or recreate
verify_result = subprocess.run(
    ["ssh", "-o", "StrictHostKeyChecking=no", f"cc@{FLOATING_IP}",
     "rclone lsd chi_s3: 2>&1 || echo 'rclone not installed — run: sudo apt-get install -y rclone'"],
    capture_output=True, text=True
)
print("Buckets visible:", verify_result.stdout)

## S3 Environment Variables

Set these environment variables before running any container or Python script that accesses object storage.

Add them to your `.env` file on the VM, or export them in your shell session.

In [ ]:
# Task 3: Print S3 env var exports for containers and scripts
# Replaced by Chameleon native S3 (CHI@TACC)

BUCKET_NAME = "training-module-proj03"

print("# Copy these exports into your shell or .env file on the VM:")
print()
print("export AWS_ACCESS_KEY_ID=<your-EC2-access-key>")
print("export AWS_SECRET_ACCESS_KEY=<your-EC2-secret-key>")
print(f"export S3_ENDPOINT_URL=https://chi.tacc.chameleoncloud.org:7480")
print(f"export S3_BUCKET={BUCKET_NAME}")
print()
print(f"# BUCKET_NAME python variable: '{BUCKET_NAME}'")
print("# Use BUCKET_NAME wherever the bucket name appears in Python code.")

## Smoke Test: Verify S3 Access

Verifies credentials and network connectivity from KVM@TACC to CHI@TACC object storage.

**Set the env vars from Task 3 first, or fill in the values below.**

In [ ]:
# Task 4: Smoke test — verify boto3 connectivity to Chameleon S3
# Replaced by Chameleon native S3 (CHI@TACC)
# Run this cell from inside the VM (or with env vars set pointing to the VM's env)

import os
import boto3
from botocore.client import Config

BUCKET_NAME = "training-module-proj03"

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ.get("S3_ENDPOINT_URL", "https://chi.tacc.chameleoncloud.org:7480"),
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID", ""),
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY", ""),
    config=Config(signature_version="s3v4"),
)

try:
    resp = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="data_arm9337/", MaxKeys=10)
    contents = resp.get("Contents", [])
    if contents:
        print(f"First {len(contents)} keys in '{BUCKET_NAME}/data_arm9337/':")
        for obj in contents:
            print(f"  {obj['Key']}")
    else:
        print(f"Bucket '{BUCKET_NAME}' is empty.")
    print(f"\nS3 connectivity OK — endpoint: {os.environ.get('S3_ENDPOINT_URL', 'https://chi.tacc.chameleoncloud.org:7480')}")
except Exception as e:
    print(f"ERROR: {e}")
    print("Check: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, S3_ENDPOINT_URL are set correctly.")
    print("Check: network connectivity from this machine to CHI@TACC.")